In [ ]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.0/337.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.7 MB/s eta 0:00:00


In [ ]:
!pip install transformers

In [ ]:
import os
import pandas as pd
import openai
import base64
import ast
from google.colab import drive
import numpy as np
from PIL import Image
import io

In [ ]:
# OpenAI API 키 설정
os.environ["OPENAI_API_KEY"] = "API_KEY"
client = openai.Client()

from google.colab import drive
drive.mount('/content/drive')

# 파일 경로 설정
file_path = '/content/drive/MyDrive/dataset/data_2024-07-25.csv'

# 데이터 불러오기
data = pd.read_csv(file_path)

# 첫 번째 행만 사용
data = data.iloc[33:34]

# 경로 변환 함수
def transform_path_list(path_list_str):
# 문자열을 리스트로 변환
    path_list = ast.literal_eval(path_list_str)
    # 각 경로 수정
    new_path_list = []
    for path in path_list:
        if path is not None:
            new_path = '/content/drive/My Drive/GeoAAC/' + path.replace('\\', '/')
            new_path_list.append(new_path)
            print("new_path =", new_path)
        else:
            new_path_list.append(path)
    return new_path_list

def encode_images_to_base64(path):
    encoded_images = []
    if os.path.isfile(path):
        with Image.open(path) as img:
            img_resized = img.resize((833, 833))
            buffer = io.BytesIO()
            img_resized.save(buffer, format='PNG')
            encoded_string = base64.b64encode(buffer.getvalue()).decode('utf-8')
            encoded_images.append(encoded_string)
            print(f"Encoded image for path=>{path}: {encoded_string[:100]}...")  # 처음 100자만 출력
    else:
        print(f"Warning: The path '{path}' is not a file.")
        encoded_images.append("None")
    return encoded_images

def generate_description_and_sentence(image_lists):
    descriptions = []
    prompts = []

    for path in image_lists:
        # 이미지를 Base64로 인코딩
        base64_string = encode_images_to_base64(path)
        if base64_string == "None":
            continue
        # 이미지 설명 추가
        prompts.append(f"data:image/jpeg;base64,{base64_string}")

    messages = [
        {"role": "system", "content": "You are a Symbolic Image Interpreter, translating images into Korean. Your task is to interpret a series of symbolic images, each representing distinct words that together form a complete sentence. Your role involves analyzing each image to identify the word it represents and then combining these insights to reconstruct the original sentence. Be precise and analytical in identifying each symbol's meaning, and ensure your final sentence accurately reflects the sequence and content of the imagery provided. Do not remember previously requested images."},
        {"role": "user", "content": [
            {"type": "text", "text": "내가 준 이미지들을 한국어 문장 하나로 설명해줄 수 있어?"}]}]

    for prompt in prompts:
        messages[1]["content"].append({"type": "image_url", "image_url": {"url": prompt}})

    print(messages)

    response = client.chat.completions.create(
        model="gpt-4o",  # GPT-4o 모델 사용
        messages=messages,
        max_tokens=2000,
        temperature=0.3 # 모델의 창의성을 줄이고 정확하고 체계적으로 해석해야함. 따라서 0.2 ~ 0.3이 적당함.
    )

    if response.choices:
        description = response.choices[0].message.content.strip()
        descriptions.append(description)
    else:
        descriptions.append("None")

    # 모든 이미지 리스트에 대한 설명을 반환
    return descriptions

# 결과 생성
data['description'] = data['ex seq3 img'].apply(transform_path_list)
data['generated_sentence'] = data['description'].apply(generate_description_and_sentence)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_seq_items', None)
np.set_printoptions(threshold=np.inf)

# 결과 출력
print(data[['description', 'generated_sentence']])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
new_path = /content/drive/My Drive/GeoAAC/image/symbol/builtin/건강/접수, 수납/돼요.png
Encoded image for path=>/content/drive/My Drive/GeoAAC/image/symbol/builtin/건강/접수, 수납/돼요.png: iVBORw0KGgoAAAANSUhEUgAAA0EAAANBCAYAAAAm/bXZAABSIUlEQVR4nO3dS4xd9Z0v+l8dZUacqjsyMpLLV0YyUjd2HcmZAFaV...
[{'role': 'system', 'content': "You are a Symbolic Image Interpreter, translating images into Korean. Your task is to interpret a series of symbolic images, each representing distinct words that together form a complete sentence. Your role involves analyzing each image to identify the word it represents and then combining these insights to reconstruct the original sentence. Be precise and analytical in identifying each symbol's meaning, and ensure your final sentence accurately reflects the sequence and content of the imagery provided. Do not remember previously requested images."}, {'r

In [ ]:
# 결과를 새로운 CSV 파일에 저장
result_file_path = '/content/drive/MyDrive/dataset/processed_results.csv'
data.to_csv(result_file_path, index=False)

print(f"Results saved to {result_file_path}")

Results saved to /content/drive/MyDrive/dataset/processed_results.csv
